In [1]:
import pandas as pd
import numpy as np

# Constants from your provided documents
FIXED_WACC = 0.3677  # From your Summary Image
TAX_RATE = 0.30     # Requested tax rate
CAP_GAINS_TAX = 0.37 # From PDF Policy [cite: 18]
SURPLUS_VALUE = 1.40 # Policy: Sell 40% higher than book [cite: 17]

def generate_forecast(project_name):
    years = np.array([1, 2, 3, 4, 5])

    if project_name == "JUSTY":
        # Data Extraction [cite: 12]
        income = np.array([52000+5000+12000, 58000+8000+18000, 72000+10000+20000, 88000+12000+25000, 93000+14000+29000])
        costs = np.array([8000+5000+6000+5000, 10000+8000+9000+9000, 14000+11000+12000+12000, 16000+13000+14000+15000, 18000+17000+16000+18000])
        initial_inv = 5000+8000+10000+15000+6000+4000+16000+8500+9000 #
        dep_amort = (5000/10) + (10000/10) + (6000/4) + (4000/4) + (16000/2.5) + (8500/8.5) + (9000/5) #
        replacement = (19000, 3) # Eq A replacement at Year 3
    else:
        # Data Extraction
        income = np.array([55000+4000+11000, 62000+7000+15000, 95000+9000+30000, 92000+15000+25000, 75000+12000+20000])
        costs = np.array([10000+5000+8000+9000, 12000+8000+10000+12000, 20000+12000+20000+16000, 16000+17000+17000+23000, 22000+20000+12000+20000])
        initial_inv = 12500+5000+9000+5000+12000+20000+10000+8000 #
        dep_amort = (5000/10) + (5000/10) + (12000/3) + (20000/3.5) + (10000/4) + (8000/8) #
        replacement = (22000, 4) # Eq A replacement at Year 4

    # Calculate Flow
    ebitda = income - costs
    ebit = ebitda - dep_amort
    taxes = np.where(ebit > 0, ebit * TAX_RATE, 0)
    net_income = ebit - taxes
    fcf = net_income + dep_amort
    fcf[replacement[1]-1] -= replacement[0] # CapEx Reinvestment

    # NPV & CAGR
    df = [1 / (1 + FIXED_WACC)**t for t in years]
    npv = np.sum(fcf * df) - initial_inv
    cagr = (income[-1] / income[0])**(1/4) - 1

    return pd.DataFrame({
        "Year": years, "Income": income, "Costs": costs,
        "Dep_Amort": [dep_amort]*5, "Net_Income": net_income, "FCF": fcf
    }), npv, cagr

# Processing
df_j, npv_j, cagr_j = generate_forecast("JUSTY")
df_a, npv_a, cagr_a = generate_forecast("AUTOMASTER")

# Export to Excel
with pd.ExcelWriter("Corporate_Finance_Analysis.xlsx", engine='openpyxl') as writer:
    df_j.to_excel(writer, sheet_name="JUSTY")
    df_a.to_excel(writer, sheet_name="AUTOMASTER")
    summary = pd.DataFrame({
        "Project": ["JUSTY", "AUTOMASTER"],
        "NPV": [npv_j, npv_a],
        "CAGR (Income)": [f"{cagr_j:.2%}", f"{cagr_a:.2%}"]
    })
    summary.to_excel(writer, sheet_name="Final_Comparison")

print("Calculations complete. Check 'Corporate_Finance_Analysis.xlsx'.")

Calculations complete. Check 'Corporate_Finance_Analysis.xlsx'.
